In [1]:
!pip install -q --no-deps --force-reinstall "timm>=1.0.20"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.4 MB/s eta 0:00:00a 0:00:01


In [2]:
import os, gc, json, time
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image
from timm.data import resolve_data_config

assert timm.__version__ >= '1.0.20', f'timm {timm.__version__} -- restart the session'
print('timm', timm.__version__, '|', len(timm.list_models('*dinov3*')), 'dinov3 models')

DATA_DIR   = '/kaggle/input/csiro-biomass'
OUT_DIR    = '/kaggle/working'

MODEL_NAME = 'vit_huge_plus_patch16_dinov3.lvd1689m'
IMG_SIZE   = 800
BATCH_SIZE = 2          # ViT-H+ @800px on 16GB. raise to 4 if memory allows.
NUM_WORKERS = 2

device = torch.device('cuda')
print('device:', torch.cuda.get_device_name(0))
print('free VRAM: %.1f GB' % (torch.cuda.mem_get_info()[0] / 1e9))

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

timm 1.0.28 | 12 dinov3 models
device: Tesla P100-PCIE-16GB
free VRAM: 16.8 GB


In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
train['base_id'] = train['sample_id'].str.split('__').str[0]

images = (train.groupby('base_id')['image_path'].first()
               .sort_index()               # deterministic order, locked forever
               .reset_index())

missing = [p for p in images['image_path'] if not os.path.exists(f'{DATA_DIR}/{p}')]
assert not missing, f'{len(missing)} image paths do not exist: {missing[:5]}'

print(f'{len(train)} target rows -> {len(images)} unique photographs')
print('all image paths verified on disk')

probe = Image.open(f"{DATA_DIR}/{images['image_path'].iloc[0]}")
print('native image size:', probe.size, probe.mode)

1785 target rows -> 357 unique photographs
all image paths verified on disk
native image size: (2000, 1000) RGB


In [4]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0,
                          img_size=IMG_SIZE)
model.eval().to(device)
for p in model.parameters():
    p.requires_grad_(False)

cfg  = resolve_data_config(model=model)
MEAN = torch.tensor(cfg['mean']).view(1, 3, 1, 1).to(device)
STD  = torch.tensor(cfg['std']).view(1, 3, 1, 1).to(device)

DIM      = model.num_features
N_PREFIX = model.num_prefix_tokens

print(f'{MODEL_NAME}')
print(f'  params      {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'  feature dim {DIM}')
print(f'  blocks      {len(model.blocks)}')
print(f'  prefix tok  {N_PREFIX}  (1 CLS + {N_PREFIX-1} register)')
print(f'  mean        {[round(x,4) for x in cfg["mean"]]}')
print(f'  std         {[round(x,4) for x in cfg["std"]]}')
print(f'  grid        {IMG_SIZE//16}x{IMG_SIZE//16} = {(IMG_SIZE//16)**2} patch tokens')

model.safetensors:   0%|          | 0.00/3.36G [00:00<?, ?B/s]

vit_huge_plus_patch16_dinov3.lvd1689m
  params      840.5M
  feature dim 1280
  blocks      32
  prefix tok  5  (1 CLS + 4 register)
  mean        [0.485, 0.456, 0.406]
  std         [0.229, 0.224, 0.225]
  grid        50x50 = 2500 patch tokens


In [5]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

class ImageOnly(Dataset):
    def __init__(self, df, size):
        self.paths = df['image_path'].tolist()
        self.size  = size
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        # no try/except -- if a read fails the job must crash, not train on grey
        img = Image.open(f'{DATA_DIR}/{self.paths[i]}').convert('RGB')
        img = img.resize((self.size, self.size), Image.BICUBIC)
        return TF.pil_to_tensor(img)          # uint8 CHW

loader = DataLoader(ImageOnly(images, IMG_SIZE), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'{len(loader)} batches of {BATCH_SIZE}')

179 batches of 2


In [6]:
FLIPS = {
    'id': lambda t: t,
    'h' : lambda t: torch.flip(t, [3]),
    'v' : lambda t: torch.flip(t, [2]),
    'hv': lambda t: torch.flip(t, [2, 3]),
}

feats = {k: np.zeros((len(images), 2 * DIM), dtype=np.float16) for k in FLIPS}
t0 = time.time()

with torch.no_grad():
    for fname, flip in FLIPS.items():
        row = 0
        for bi, batch in enumerate(loader):
            x = batch.to(device, non_blocking=True).float().div_(255.)
            x = (x - MEAN) / STD
            x = flip(x)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                tokens = model.forward_features(x)        # (B, 2505, DIM)
            cls   = tokens[:, 0]
            patch = tokens[:, N_PREFIX:].mean(dim=1)
            vec   = torch.cat([cls, patch], dim=1).float().cpu().numpy()
            feats[fname][row:row + len(vec)] = vec.astype(np.float16)
            row += len(vec)
            if bi % 40 == 0:
                print(f'  {fname} {bi:4d}/{len(loader)}  '
                      f'{time.time()-t0:6.0f}s', flush=True)
        assert row == len(images), (row, len(images))
        print(f'view {fname} done  ({time.time()-t0:.0f}s elapsed)', flush=True)

print(f'\ntotal {time.time()-t0:.0f}s')

  id    0/179       3s
  id   40/179      73s
  id   80/179     143s
  id  120/179     213s
  id  160/179     284s
view id done  (314s elapsed)
  h    0/179     316s
  h   40/179     387s
  h   80/179     457s
  h  120/179     528s
  h  160/179     598s
view h done  (629s elapsed)
  v    0/179     631s
  v   40/179     701s
  v   80/179     771s
  v  120/179     842s
  v  160/179     912s
view v done  (943s elapsed)
  hv    0/179     945s
  hv   40/179    1015s
  hv   80/179    1086s
  hv  120/179    1156s
  hv  160/179    1227s
view hv done  (1257s elapsed)

total 1257s


In [7]:
stack = np.stack([feats[k] for k in FLIPS], axis=1)   # (n_img, 4, 2*DIM)
np.save(f'{OUT_DIR}/features.npy', stack)
images['base_id'].to_frame().to_csv(f'{OUT_DIR}/feature_ids.csv', index=False)

manifest = {
    'model': MODEL_NAME, 'img_size': IMG_SIZE, 'feature_dim': int(DIM),
    'pooled': ['cls', 'patch_mean'], 'vector_dim': int(2 * DIM),
    'views': list(FLIPS), 'n_images': int(len(images)),
    'mean': [float(x) for x in cfg['mean']], 'std': [float(x) for x in cfg['std']],
    'shape': list(stack.shape), 'seconds': round(time.time() - t0, 1),
}
with open(f'{OUT_DIR}/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print(f'\nfeatures.npy  {stack.nbytes/1e6:.1f} MB  shape {stack.shape}')
print('sanity -- feature std per view:',
      [round(float(stack[:, i].astype(np.float32).std()), 4) for i in range(4)])
assert stack.astype(np.float32).std() > 1e-3, 'features are constant -- model is broken'
print('\nDONE. Attach this notebook output as input to notebook 2.')

{
  "model": "vit_huge_plus_patch16_dinov3.lvd1689m",
  "img_size": 800,
  "feature_dim": 1280,
  "pooled": [
    "cls",
    "patch_mean"
  ],
  "vector_dim": 2560,
  "views": [
    "id",
    "h",
    "v",
    "hv"
  ],
  "n_images": 357,
  "mean": [
    0.485,
    0.456,
    0.406
  ],
  "std": [
    0.229,
    0.224,
    0.225
  ],
  "shape": [
    357,
    4,
    2560
  ],
  "seconds": 1257.5
}

features.npy  7.3 MB  shape (357, 4, 2560)
sanity -- feature std per view: [0.2169, 0.2168, 0.2166, 0.2164]

DONE. Attach this notebook output as input to notebook 2.
